In [5]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

# ──────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────
ATAC_DIR       = "/data1/lukszam/Shayan/DL/DLProject/atac_chr19_subsampled"
MODEL_SAVE_DIR = "/data1/lukszam/Shayan/DL/"
SPARSITY       = 50
BATCH_SIZE     = 8
NUM_WORKERS    = 2
INPUT_DIM      = 109792

FINAL_EPOCHS   = 500
FINAL_PATIENCE = 20
MIN_DELTA      = 1.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


# ──────────────────────────────────────────────
# DATASET
# ──────────────────────────────────────────────
class ATACDataset(Dataset):
    def __init__(self, atac_dir, sparsity, sparse_function=None, file_list=None):
        self.atac_dir        = atac_dir
        self.sparsity        = sparsity
        self.sparse_function = sparse_function
        self.files           = file_list if file_list else sorted(os.listdir(atac_dir))

    def _load_tsv(self, path):
        df  = pd.read_csv(path, sep="\t")
        col = [df.columns[1]]
        return df[col]

    def _load_sparse_tsv(self, path):
        df   = pd.read_csv(path, sep="\t")
        cols = [i for i in df.columns if f"_{self.sparsity}_" in i]
        return df[cols]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname     = self.files[idx]
        atac_path = os.path.join(self.atac_dir, fname)
        dense_df  = self._load_tsv(atac_path)
        y         = dense_df[dense_df.columns[0]].values.astype(np.float32)
        if self.sparse_function is not None:
            x = self.sparse_function(y, self.sparsity)
        else:
            sparse_df   = self._load_sparse_tsv(atac_path)
            sparse_cols = sparse_df.columns[:]
            col         = np.random.choice(sparse_cols)
            x           = sparse_df[col].values.astype(np.float32)
        return torch.from_numpy(x), torch.from_numpy(y)


def create_dataloader(atac_dir, batch_size=4, shuffle=True, num_workers=4,
                      sparsity=1, val_split=0.2, seed=42):
    all_files = sorted(os.listdir(atac_dir))
    rng       = np.random.RandomState(seed)
    rng.shuffle(all_files)
    split_idx   = int(len(all_files) * (1 - val_split))
    train_files = all_files[:split_idx]
    val_files   = all_files[split_idx:]

    train_dataset = ATACDataset(atac_dir=atac_dir, sparsity=sparsity,
                                sparse_function=None, file_list=train_files)
    val_dataset   = ATACDataset(atac_dir=atac_dir, sparsity=sparsity,
                                sparse_function=None, file_list=val_files)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=shuffle,
                              num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader


# ──────────────────────────────────────────────
# MODEL
# ──────────────────────────────────────────────
class AttentionGate(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, hidden_dim):
        super().__init__()
        self.W_e     = nn.Linear(encoder_dim, hidden_dim)
        self.W_d     = nn.Linear(decoder_dim, hidden_dim)
        self.v       = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, e, d):
        e_proj = self.W_e(e)
        d_proj = self.W_d(d)
        attn   = self.v(torch.tanh(e_proj + d_proj))
        attn   = self.sigmoid(attn)
        return e * attn


class VAE(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dims=None,
                 latent_dim=128, dropout=0.1, decode_alpha=1):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [4096, 1024, 256]

        self.dropout         = dropout
        self.decode_alpha    = decode_alpha
        self.attention_gates = nn.ModuleList()

        encoder_layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            encoder_layers.append(nn.Linear(prev_dim, h))
            encoder_layers.append(nn.BatchNorm1d(h))
            encoder_layers.append(nn.ReLU())
            encoder_layers.append(nn.Dropout(self.dropout))
            prev_dim = h
        self.encoder   = nn.Sequential(*encoder_layers)
        self.fc_mu     = nn.Linear(prev_dim, latent_dim)
        self.fc_logvar = nn.Linear(prev_dim, latent_dim)

        decoder_layers = []
        prev_dim = latent_dim
        for h in reversed(hidden_dims):
            decoder_layers.append(nn.Linear(prev_dim, h))
            decoder_layers.append(nn.BatchNorm1d(h))
            decoder_layers.append(nn.ReLU())
            prev_dim = h
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        self.decoder = nn.Sequential(*decoder_layers)

        decoder_dims = list(reversed(hidden_dims))
        for e_dim, d_dim in zip(decoder_dims, decoder_dims):
            self.attention_gates.append(
                AttentionGate(e_dim, d_dim, hidden_dim=d_dim // 2)
            )

    def encode(self, x):
        activations = []
        h = x
        for layer in self.encoder:
            h = layer(h)
            if isinstance(layer, nn.ReLU):
                activations.append(h)
        mu     = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar, activations

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, activations):
        gate_idx = 0
        for i, layer in enumerate(self.decoder[:-1]):
            z = layer(z)
            if isinstance(layer, nn.Linear) and gate_idx < len(self.attention_gates):
                skip       = activations[-(gate_idx + 1)]
                gated_skip = self.attention_gates[gate_idx](skip, z)
                z          = z + self.decode_alpha * gated_skip
                gate_idx  += 1
        return self.decoder[-1](z)

    def forward(self, x):
        mu, logvar, activations = self.encode(x)
        z     = self.reparameterize(mu, logvar)
        x_hat = self.decode(z, activations)
        return x_hat, mu, logvar


# ──────────────────────────────────────────────
# LOSS
# ──────────────────────────────────────────────
def vae_loss(x_hat, x, mu, logvar, beta=1.0):
    recon_loss = F.mse_loss(x_hat, x, reduction='mean')
    kl         = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl, recon_loss, kl


# ──────────────────────────────────────────────
# TRAIN LOOP
# ──────────────────────────────────────────────
def train_vae(model, train_loader, val_loader, device, model_save_dir,
              epochs=500, lr=1e-3, beta_start=0.0, beta_end=1.0,
              beta_warmup=100, patience=20, min_delta=1.0,
              save_name="best_vae_final.pt"):

    optimizer        = Adam(model.parameters(), lr=lr)
    best_val_loss    = float("inf")
    patience_counter = 0

    history = {"train_loss": [], "val_loss": [],
               "recon_loss": [], "val_recon_loss": [],  # val recon now tracked
               "kl_loss":    [], "beta": []}

    for epoch in range(epochs):
        model.train()
        total_loss = total_recon = total_kl = 0

        if epoch < beta_warmup:
            beta = beta_start + (beta_end - beta_start) * (epoch / beta_warmup)
        else:
            beta = beta_end

        for x, y in train_loader:
            x = x.to(device).float()
            y = y.to(device).float()
            x_hat, mu, logvar    = model(x)
            loss, recon_loss, kl = vae_loss(x_hat, y, mu, logvar, beta=beta)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) #newwww line
            optimizer.step()
            total_loss  += loss.item()
            total_recon += recon_loss.item()
            total_kl    += kl.item()

        avg_loss  = total_loss  / len(train_loader)
        avg_recon = total_recon / len(train_loader)
        avg_kl    = total_kl    / len(train_loader)

        # ── val loop — now tracks recon separately ──
        model.eval()
        val_loss  = 0
        val_recon = 0
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device).float()
                y = y.to(device).float()
                x_hat, mu, logvar   = model(x)
                loss, recon_loss, _ = vae_loss(x_hat, y, mu, logvar, beta=beta)
                val_loss  += loss.item()
                val_recon += recon_loss.item()

        avg_val_loss  = val_loss  / len(val_loader)
        avg_val_recon = val_recon / len(val_loader)

        history["train_loss"].append(avg_loss)
        history["val_loss"].append(avg_val_loss)
        history["recon_loss"].append(avg_recon)
        history["val_recon_loss"].append(avg_val_recon)
        history["kl_loss"].append(avg_kl)
        history["beta"].append(beta)

        print(
            f"Epoch {epoch+1:03d} | "
            f"Train Loss: {avg_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f} | "
            f"Val Recon: {avg_val_recon:.4f} | "
            f"KL: {avg_kl:.4f} | "
            f"Beta: {beta:.4f}"
        )

        if avg_val_loss < best_val_loss - min_delta:
            best_val_loss    = avg_val_loss
            patience_counter = 0
            torch.save(model.state_dict(),
                       os.path.join(model_save_dir, save_name))
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

    return history, best_val_loss


# ──────────────────────────────────────────────
# PLOTS
# ──────────────────────────────────────────────
BG    = "#FAFAF8"
MUTED = "#73726C"
GCOL  = "#E8E6DF"
C = {
    "blue":   "#3266AD", "teal":   "#1D9E75", "amber":  "#BA7517",
    "coral":  "#D85A30", "purple": "#7F77DD", "gray":   "#888780",
    "red":    "#E24B4A", "green":  "#639922",
}

plt.rcParams.update({
    "font.family":       "sans-serif",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.color":        GCOL,
    "grid.linewidth":    0.5,
    "axes.facecolor":    BG,
    "figure.facecolor":  BG,
})


def _annotate_min(ax, epochs, values, color):
    clean = [(i, v) for i, v in enumerate(values)
             if v is not None and np.isfinite(v) and v > 0]
    if not clean:
        return
    idx, best = min(clean, key=lambda x: x[1])
    ax.scatter([epochs[idx]], [best], color=color, s=60, zorder=5,
               edgecolors="white", linewidths=1.2)
    ax.annotate(f"min {best:.2f}",
                xy=(epochs[idx], best),
                xytext=(epochs[idx] + max(len(epochs)*0.04, 2), best),
                fontsize=8, color=color,
                arrowprops=dict(arrowstyle="-", color=color, lw=0.8))


def plot_training_metrics(history, save_dir):
    epochs      = list(range(1, len(history["train_loss"]) + 1))
    train_total = history["train_loss"]
    val_total   = history["val_loss"]
    train_recon = history["recon_loss"]
    val_recon   = history["val_recon_loss"]
    kl          = history["kl_loss"]
    beta        = history["beta"]

    # log-safe versions for total loss panels
    train_log = [v if np.isfinite(v) and v > 0 else float("nan") for v in train_total]
    val_log   = [v if np.isfinite(v) and v > 0 else float("nan") for v in val_total]

    # KL spike mask
    kl_clipped = [v if v < 10 else float("nan") for v in kl]

    fig = plt.figure(figsize=(18, 10), facecolor=BG, tight_layout=True)
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.32)

    def _ax(pos, title, ylabel="Loss"):
        a = fig.add_subplot(pos)
        a.set_title(title, fontsize=12, pad=8)
        a.set_xlabel("Epoch", fontsize=10, color=MUTED)
        a.set_ylabel(ylabel, fontsize=10, color=MUTED)
        return a

    # 1 — train total loss log scale
    ax = _ax(gs[0, 0], "Train total loss  (log scale)")
    ax.plot(epochs, train_log, color=C["blue"], lw=1.8, label="Train")
    ax.set_yscale("log")
    ax.set_ylabel("Loss (log scale)", fontsize=10, color=MUTED)
    ax.legend(fontsize=9, frameon=False)
    _annotate_min(ax, epochs, train_log, C["blue"])

    # 2 — val total loss log scale
    ax = _ax(gs[0, 1], "Val total loss  (log scale)")
    ax.plot(epochs, val_log, color=C["coral"], lw=1.8, linestyle="--", label="Val")
    ax.set_yscale("log")
    ax.set_ylabel("Loss (log scale)", fontsize=10, color=MUTED)
    ax.legend(fontsize=9, frameon=False)
    _annotate_min(ax, epochs, val_log, C["coral"])

    # 3 — train recon vs val recon (THE key plot)
    ax = _ax(gs[0, 2], "Reconstruction loss  (MSE)  —  train vs val")
    ax.plot(epochs, train_recon, color=C["teal"],  lw=1.8, label="Train recon")
    ax.plot(epochs, val_recon,   color=C["coral"], lw=1.8,
            linestyle="--", label="Val recon")
    ax.legend(fontsize=9, frameon=False)
    _annotate_min(ax, epochs, train_recon, C["teal"])
    _annotate_min(ax, epochs, val_recon,   C["coral"])

    # 4 — KL divergence
    ax = _ax(gs[1, 0], "KL divergence", ylabel="KL")
    ax.plot(epochs, kl_clipped, color=C["purple"], lw=1.8)
    ax.set_ylim(0, 8)

    # 5 — beta annealing
    ax = _ax(gs[1, 1], "β annealing schedule", ylabel="β")
    ax.plot(epochs, beta, color=C["amber"], lw=1.8)
    ax.fill_between(epochs, 0, beta, color=C["amber"], alpha=0.12)

    # 6 — val recon - train recon gap
    gap = [v - t for v, t in zip(val_recon, train_recon)]
    ax = _ax(gs[1, 2], "Val − Train reconstruction gap", ylabel="Δ MSE")
    ax.plot(epochs, gap, color=C["red"], lw=1.8)
    ax.axhline(0, color=MUTED, lw=0.8, linestyle="--")
    ax.fill_between(epochs, 0, gap,
                    where=[g > 0 for g in gap],
                    color=C["red"], alpha=0.12, label="Val > Train")
    ax.legend(fontsize=9, frameon=False)

    fig.suptitle("Final training — ATAC-seq VAE", fontsize=15, y=1.01, color="#2C2C2A")
    path = os.path.join(save_dir, "training_metrics_final.png")
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close(fig)
    print(f"Saved → {path}")


# ──────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────
if __name__ == "__main__":
    os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

    # load best params from Optuna search
    params_path = os.path.join(MODEL_SAVE_DIR, "best_params.json")
    with open(params_path) as f:
        best = json.load(f)

    print("Loaded best params:")
    for k, v in best.items():
        print(f"  {k}: {v}")

    train_loader, val_loader = create_dataloader(
        atac_dir    = ATAC_DIR,
        batch_size  = BATCH_SIZE,
        num_workers = NUM_WORKERS,
        sparsity    = SPARSITY,
    )

    model = VAE(
        input_dim    = INPUT_DIM,
        hidden_dims  = best["hidden_dims"],
        latent_dim   = best["latent_dim"],
        dropout      = best["dropout"],
        decode_alpha = best["decode_alpha"],
    ).to(DEVICE)

    history, _ = train_vae(
        model,
        train_loader,
        val_loader,
        device         = DEVICE,
        model_save_dir = MODEL_SAVE_DIR,
        epochs         = FINAL_EPOCHS,
        lr             = best["lr"],
        beta_end       = best["beta_end"],
        beta_warmup    = best["beta_warmup"],
        patience       = FINAL_PATIENCE,
        min_delta      = MIN_DELTA,
        save_name      = "best_vae_final.pt",
    )

    with open(os.path.join(MODEL_SAVE_DIR, "training_history_final.json"), "w") as f:
        json.dump(history, f, indent=2)

    plot_training_metrics(history, MODEL_SAVE_DIR)

    print("\nDone. Outputs in", MODEL_SAVE_DIR)
    print("  best_vae_final.pt")
    print("  training_metrics_final.png")
    print("  training_history_final.json")

Using device: cuda
Loaded best params:
  hidden_dims: [4096, 1024, 256]
  latent_dim: 64
  lr: 0.004907249536449826
  dropout: 0.39964876108980285
  beta_end: 1.3096981611494822
  beta_warmup: 29
  decode_alpha: 1.426754462524831
Epoch 001 | Train Loss: 2630.2808 | Val Loss: nan | Val Recon: inf | KL: 0.2189 | Beta: 0.0000
Epoch 002 | Train Loss: 2410.9774 | Val Loss: 9283178163835089457630140170240.0000 | Val Recon: 9246004903807759225137099964416.0000 | KL: 0.2731 | Beta: 0.0452
Epoch 003 | Train Loss: 2206.5817 | Val Loss: 703376080912802113388544.0000 | Val Recon: 699727372581097594421248.0000 | KL: 0.3726 | Beta: 0.0903
Epoch 004 | Train Loss: 2010.9196 | Val Loss: 3241177205702656.0000 | Val Recon: 380643959635968.0000 | KL: 0.5364 | Beta: 0.1355
Epoch 005 | Train Loss: 1821.9839 | Val Loss: 1391563767808.0000 | Val Recon: 1365789507584.0000 | KL: 0.6445 | Beta: 0.1806
Epoch 006 | Train Loss: 1719.0559 | Val Loss: 530897920.0000 | Val Recon: 519610208.0000 | KL: 0.7462 | Beta: 0.